# Session 2 — Forward Kinematics & Tool Center Point (Meca500)

Ce notebook implémente et vérifie la **cinématique directe symbolique (SymPy)** du Mecademic Meca500 selon les deux méthodes du cours :
1. **Placement simple des repères** (*Simple Frame Placement*, Session 2 Fig. 2.2) + alignement sur le repère bride constructeur (`FRF`).
2. **Paramètres Denavit–Hartenberg modifiés (Craig)** (Cours *Forward Kinematics*, Slide 45).

Il importe les fonctions modulaires depuis `src/ForwardKinematics.py` pour faciliter la réutilisation dans le rapport et en Session 3.

In [2]:
import sys
from pathlib import Path

import numpy as np
import sympy as sp
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ForwardKinematics import (
    MECA500_PARAMS,
    SESSION2_TEST_CONFIGS,
    Meca500Kinematics,
    build_symbolic_fk_dh,
    build_symbolic_fk_simple,
    build_trf_matrix,
)

sp.init_printing(use_unicode=True)
print("Dimensions Meca500 [mm]:", MECA500_PARAMS)

Dimensions Meca500 [mm]: {'l_0': 135.0, 'l_2': 135.0, 'l_3': 38.0, 'l_4': 120.0, 'l_6': 70.0}


## 1. Dérivation symbolique & Équivalence Simple Frame vs Craig D-H

À $\Theta = [0,0,0,0,0,0]$, les repères de la Figure 2.2 ont tous $+X$ vers l'avant et $+Z$ vers le haut.
Pour correspondre au repère bride officiel Mecademic (**FRF** : $+Z_{\text{FRF}}$ sortant le long de l'axe 6, $+Y_{\text{FRF}} \parallel +Y_{\text{BRF}}$, $+X_{\text{FRF}}$ vers le bas), on applique une rotation finale $R_Y(\pi/2)$.

In [ ]:
transforms_simple, T_simple_frf = build_symbolic_fk_simple(align_to_frf=True)
links_dh, T_dh_frf = build_symbolic_fk_dh(align_to_frf=True)

for name, mat in transforms_simple.items():
    print(f"{name}:")
    display(mat)

is_equal = sp.simplify(T_simple_frf - T_dh_frf) == sp.zeros(4, 4)
print("Équivalence symbolique Simple Frame == Craig D-H (aligné FRF) :", is_equal)

T_B_0:


⎡1  0  0  0 ⎤
⎢           ⎥
⎢0  1  0  0 ⎥
⎢           ⎥
⎢0  0  1  l₀⎥
⎢           ⎥
⎣0  0  0  1 ⎦

T_0_1:


⎡cos(θ₁)  -sin(θ₁)  0  0⎤
⎢                       ⎥
⎢sin(θ₁)  cos(θ₁)   0  0⎥
⎢                       ⎥
⎢   0        0      1  0⎥
⎢                       ⎥
⎣   0        0      0  1⎦

T_1_2:


⎡cos(θ₂)   0  sin(θ₂)  0⎤
⎢                       ⎥
⎢   0      1     0     0⎥
⎢                       ⎥
⎢-sin(θ₂)  0  cos(θ₂)  0⎥
⎢                       ⎥
⎣   0      0     0     1⎦

... (9 outputs left out)

⎡1  0  0  l₆⎤
⎢           ⎥
⎢0  1  0  0 ⎥
⎢           ⎥
⎢0  0  1  0 ⎥
⎢           ⎥
⎣0  0  0  1 ⎦

T_T_FRF:


⎡0   0  1  0⎤
⎢           ⎥
⎢0   1  0  0⎥
⎢           ⎥
⎢-1  0  0  0⎥
⎢           ⎥
⎣0   0  0  1⎦

Équivalence symbolique Simple Frame == Craig D-H (aligné FRF) : True


## 2. Évaluation des 7 configurations (Section 2.2)

Convention d'orientation Mecademic : **angles d'Euler intrinsèques (mobiles) $XYZ$** ($R = R_X(\alpha)\,R_Y(\beta)\,R_Z(\gamma)$), avec $\alpha = 0^\circ$ lorsque $|\beta| = 90^\circ$ (gimbal lock).

In [4]:
kin_frf = Meca500Kinematics(method="simple", align_to_frf=True)
kin_fig22 = Meca500Kinematics(method="simple", align_to_frf=False)

print(f"{'Joints [deg]':24s} | {'SymPy (FRF Mecademic) [mm | deg]':40s} | {'SymPy (Fig. 2.2 sans Ry(90))'}")
print("-" * 105)
for q in SESSION2_TEST_CONFIGS:
    pose_frf = np.round(kin_frf.forward_pose(q), 3).tolist()
    pose_fig = np.round(kin_fig22.forward_pose(q), 3).tolist()
    print(f"{str(q):24s} | {str(pose_frf):40s} | {pose_fig}")

Joints [deg]             | SymPy (FRF Mecademic) [mm | deg]         | SymPy (Fig. 2.2 sans Ry(90))
---------------------------------------------------------------------------------------------------------
[0, 0, 0, 0, 0, 0]       | [190.0, 0.0, 308.0, 0.0, 90.0, 0.0]      | [190.0, 0.0, 308.0, 0.0, 0.0, 0.0]
[90, 0, 0, 0, 0, 0]      | [0.0, 190.0, 308.0, -90.0, 0.0, 90.0]    | [0.0, 190.0, 308.0, 0.0, 0.0, 90.0]
[0, 0, -90, 0, 0, 0]     | [-38.0, 0.0, 460.0, 0.0, 0.0, 0.0]       | [-38.0, 0.0, 460.0, 0.0, -90.0, 0.0]
[0, 90, -90, 0, 0, 0]    | [325.0, 0.0, 173.0, 0.0, 90.0, 0.0]      | [325.0, 0.0, 173.0, 0.0, 0.0, 0.0]
[0, 0, 0, 90, 0, 0]      | [190.0, 0.0, 308.0, 0.0, 90.0, 90.0]     | [190.0, 0.0, 308.0, 90.0, 0.0, 0.0]
[0, 0, 0, 0, 90, 0]      | [120.0, 0.0, 238.0, 180.0, 0.0, 180.0]   | [120.0, 0.0, 238.0, 0.0, 90.0, 0.0]
[0, 0, 0, 0, 0, 90]      | [190.0, 0.0, 308.0, 0.0, 90.0, 90.0]     | [190.0, 0.0, 308.0, 90.0, 0.0, 0.0]


## 3. Comparaison avec l'interface Web (Section 2.1 vs 2.2)

Coller ci-dessous les 7 poses relevées dans le MecaPortal pour vérifier automatiquement les écarts.

In [ ]:
# Remplir les 6 dernières lignes après relevé sur MecaPortal :
web_interface_poses = {
    (0, 0, 0, 0, 0, 0): [190.0, 0.0, 308.0, 0.0, 90.0, 0.0],
    (90, 0, 0, 0, 0, 0): [0.0, 190.0, 308.0, -90.0, 0.0, 90.0],
    (0, 0, -90, 0, 0, 0): [-38.0, 0.0, 460.0, 0.0, 0.0, 0.0],
    (0, 90, -90, 0, 0, 0): [325.0, 0.0, 173.0, 0.0, 90.0, 0.0],
    (0, 0, 0, 90, 0, 0): [190.0, 0.0, 308.0, 90.0, 90.0, 0.0],
    (0, 0, 0, 0, 90, 0): [120.0, 0.0, 238.0, -180.0, 0.0, 180.0],
    (0, 0, 0, 0, 0, 90): [190.0, 0.0, 308.0, 90.0, 90.0, 0.0],
}

for q_tuple, web_pose in web_interface_poses.items():
    sym_pose = kin_frf.forward_pose(list(q_tuple))
    err = np.round(np.array(web_pose) - sym_pose, 4)
    print(f"q={list(q_tuple)} -> écart (Web - SymPy) = {err.tolist()}")

q=[0, 0, 0, 0, 0, 0] -> écart (Web - SymPy) = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## 4. Extension Tool Offset / TCP Laser (Section 2.3 — à compléter avec les cotes exactes)

D'après `docs/LaserTool.pdf` (`mecaLaserGripper_90deg_dwg1`), le support monte le laser à **90°** par rapport à l'axe 6 avec une cote de **13.7 mm** (et une distance de travail laser de **264 mm** en Session 3). Une fois l'offset exact `(x, y, z, alpha, beta, gamma)` vérifié dans `SetTrf(...)`, tester ci-dessous :

In [ ]:
# Exemple paramétrable pour Section 2.3.2 (valeurs à confirmer au labo)
trf_params = dict(x=0.0, y=0.0, z=13.7, alpha_deg=0.0, beta_deg=90.0, gamma_deg=0.0)
T_frf_tcp = build_trf_matrix(**trf_params)
print("Pose à q = [0, 0, 0, 0, 0, 0] avec TRF exemple :", np.round(kin_frf.forward_pose([0, 0, 0, 0, 0, 0], T_flange_tcp=T_frf_tcp), 4))